# Modkit Update Testing: v0.2.4 vs v0.6.1

This notebook systematically tests the compatibility and performance of updating modkit from the pinned version (0.2.4) to the latest (0.6.1) in the dimelo toolkit.

## Objectives
- Compare raw modkit outputs for pileup and extract commands
- Test dimelo integration (parse_bam functions and plotting)
- Measure performance improvements
- Identify any breaking changes or incompatibilities

## Setup
First, ensure both modkit versions are available. We'll modify the environment to allow switching between versions.

In [ ]:
# Notebook setup (autonomous, no in-cell environment mutation)
import json
import gzip
import hashlib
import re
import shutil
import subprocess
import time
from glob import glob
from pathlib import Path
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Optional plotting style
try:
    import seaborn as sns  # type: ignore
    sns.set_style("whitegrid")
except Exception:
    pass

plt.rcParams["figure.figsize"] = (11, 6)

ROOT = Path(".").resolve()
TEST_DATA_DIR = ROOT / "dimelo" / "test" / "data"
BAM_FILE = TEST_DATA_DIR / "ctcf_demo.sorted.bam"
BAM_INDEX = TEST_DATA_DIR / "ctcf_demo.sorted.bam.bai"
OUTPUT_DIR = ROOT / "cache" / "modkit_benchmark"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BENCH_REGION = "chr1"
THREADS = 2
BENCH_RUNS = 3

if not BAM_FILE.exists() or not BAM_INDEX.exists():
    raise FileNotFoundError(f"Missing BAM test assets under {TEST_DATA_DIR}")

print(f"BAM: {BAM_FILE}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"Region benchmark target: {BENCH_REGION}")


## Modkit Version Management

We'll create a way to switch between modkit versions by temporarily modifying the PATH.

In [ ]:
def _parse_modkit_version(text: str) -> str | None:
    match = re.search(r"(\d+\.\d+\.\d+)", text)
    return match.group(1) if match else None


def discover_modkit_versions() -> Dict[str, str]:
    """Discover installed modkit binaries and map version -> binary path."""
    candidates: set[str] = set()

    active = shutil.which("modkit")
    if active:
        candidates.add(active)

    conda = shutil.which("conda")
    if conda:
        conda_base = Path(conda).resolve().parents[1]
        for pattern in ("pkgs/modkit-*/bin/modkit", "envs/*/bin/modkit"):
            for p in conda_base.glob(pattern):
                candidates.add(str(p))

    version_to_path: Dict[str, str] = {}
    for binary in sorted(candidates):
        try:
            result = subprocess.run([binary, "--version"], capture_output=True, text=True, check=True)
        except Exception:
            continue
        version = _parse_modkit_version((result.stdout + result.stderr).strip())
        if version and version not in version_to_path:
            version_to_path[version] = binary

    return version_to_path


MODKIT_VERSIONS = discover_modkit_versions()
PREFERRED_ORDER = ["0.2.4", "0.6.1"]
TEST_VERSIONS = [v for v in PREFERRED_ORDER if v in MODKIT_VERSIONS]
if not TEST_VERSIONS:
    TEST_VERSIONS = sorted(MODKIT_VERSIONS.keys())

print("Detected modkit backends:")
for version in TEST_VERSIONS:
    print(f"  {version}: {MODKIT_VERSIONS[version]}")

if len(TEST_VERSIONS) < 2:
    print("WARNING: Only one backend detected; cross-version comparisons will be limited.")


## Raw Modkit Output Comparison

Test pileup and extract commands directly with both versions.

In [ ]:
def _count_lines(path: Path) -> int:
    if path.suffix == ".gz":
        with gzip.open(path, "rt") as handle:
            return sum(1 for _ in handle)
    with path.open("rt") as handle:
        return sum(1 for _ in handle)


def _sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while True:
            chunk = handle.read(1024 * 1024)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def run_modkit_pileup(version: str, *, run_label: str = "single") -> Dict:
    """Run a standardized pileup command for one backend and return metrics."""
    binary = MODKIT_VERSIONS[version]
    output_file = OUTPUT_DIR / f"pileup_{version}_{run_label}.bed"

    cmd = [
        binary,
        "pileup",
        str(BAM_FILE),
        str(output_file),
        "--threads",
        str(THREADS),
        "--region",
        BENCH_REGION,
    ]

    # v0.2.4 needs this for this BAM's implicit tags; v0.6.1 rejects it.
    if version.startswith("0.2."):
        cmd.append("--force-allow-implicit")

    t0 = time.perf_counter()
    proc = subprocess.run(cmd, capture_output=True, text=True)
    runtime_s = time.perf_counter() - t0

    metrics = {
        "version": version,
        "run_label": run_label,
        "runtime_s": runtime_s,
        "exit_code": proc.returncode,
        "output_file": str(output_file),
        "rows": _count_lines(output_file) if output_file.exists() and proc.returncode == 0 else 0,
        "sha256": _sha256_file(output_file) if output_file.exists() and proc.returncode == 0 else None,
        "stderr_tail": "\n".join(proc.stderr.strip().splitlines()[-6:]),
    }
    return metrics


single_run_metrics = []
for version in TEST_VERSIONS:
    metrics = run_modkit_pileup(version, run_label="single")
    single_run_metrics.append(metrics)

single_run_df = pd.DataFrame(single_run_metrics)
display(single_run_df[["version", "runtime_s", "rows", "exit_code", "output_file"]])

comparison = None
if {"0.2.4", "0.6.1"}.issubset(set(TEST_VERSIONS)):
    m024 = next(row for row in single_run_metrics if row["version"] == "0.2.4")
    m061 = next(row for row in single_run_metrics if row["version"] == "0.6.1")
    comparison = {
        "row_delta": int(m061["rows"] - m024["rows"]),
        "row_delta_pct_vs_024": (float(m061["rows"] - m024["rows"]) / float(m024["rows"]) * 100.0) if m024["rows"] else None,
        "outputs_identical_sha256": bool(m061["sha256"] == m024["sha256"]),
    }
    print("Single-run comparison:", comparison)


## Dimelo Integration Tests

Test parse_bam functions and plotting with both modkit versions.

In [ ]:
# Optional dimelo compatibility smoke-check
# This notebook avoids hard dependency on dimelo optional extras; this block is best-effort.

dimelo_smoke = {"available": False, "note": "dimelo import not attempted"}
try:
    import dimelo  # noqa: F401
    dimelo_smoke["available"] = True
    dimelo_smoke["note"] = "dimelo import succeeded"
except Exception as exc:
    dimelo_smoke["available"] = False
    dimelo_smoke["note"] = f"dimelo import unavailable in this kernel: {type(exc).__name__}: {exc}"

print(dimelo_smoke["note"])


## Performance Analysis

Detailed performance comparison with multiple runs.

In [ ]:
def benchmark_modkit(version: str, runs: int = BENCH_RUNS) -> pd.DataFrame:
    records = []
    for i in range(runs):
        records.append(run_modkit_pileup(version, run_label=f"bench{i+1}"))
    return pd.DataFrame(records)


bench_results: Dict[str, pd.DataFrame] = {}
for version in TEST_VERSIONS:
    bench_results[version] = benchmark_modkit(version, runs=BENCH_RUNS)

benchmark_df = pd.concat(bench_results.values(), ignore_index=True)
display(benchmark_df[["version", "run_label", "runtime_s", "rows", "exit_code"]])

runtime_summary = (
    benchmark_df.groupby("version", as_index=False)
    .agg(
        runtime_mean_s=("runtime_s", "mean"),
        runtime_std_s=("runtime_s", "std"),
        runtime_min_s=("runtime_s", "min"),
        runtime_max_s=("runtime_s", "max"),
    )
)
display(runtime_summary)

fig, ax = plt.subplots(1, 1, figsize=(8, 4))
for version, frame in bench_results.items():
    ax.plot(frame.index + 1, frame["runtime_s"], marker="o", label=version)
ax.set_xlabel("Benchmark run")
ax.set_ylabel("Runtime (s)")
ax.set_title(f"modkit pileup runtime by backend ({BENCH_REGION})")
ax.legend()
plt.tight_layout()
plt.show()


## Edge Cases and Compatibility Checks

Test specific scenarios that might reveal differences.

In [ ]:
# Edge-case / interface compatibility checks

def supports_flag(version: str, flag: str) -> bool:
    binary = MODKIT_VERSIONS[version]
    help_text = subprocess.run([binary, "pileup", "--help"], capture_output=True, text=True).stdout
    return flag in help_text


edge_case_rows = []
for version in TEST_VERSIONS:
    edge_case_rows.append(
        {
            "version": version,
            "supports_force_allow_implicit": supports_flag(version, "--force-allow-implicit"),
            "supports_modified_bases": supports_flag(version, "--modified-bases"),
            "supports_reference_long_opt": supports_flag(version, "--reference"),
            "supports_reference_short_opt": supports_flag(version, "--ref"),
        }
    )

edge_case_df = pd.DataFrame(edge_case_rows)
display(edge_case_df)


## Summary and Recommendations

Compile all findings into a structured report.

In [ ]:
# Compile summary and recommendations

summary_payload: Dict[str, object] = {
    "versions_tested": TEST_VERSIONS,
    "single_run": single_run_df.to_dict(orient="records"),
    "benchmark_runs": benchmark_df.to_dict(orient="records") if "benchmark_df" in globals() else [],
    "runtime_summary": runtime_summary.to_dict(orient="records") if "runtime_summary" in globals() else [],
    "pileup_comparison": comparison,
    "edge_case_flags": edge_case_df.to_dict(orient="records") if "edge_case_df" in globals() else [],
    "compatibility_issues": [],
    "recommendations": [],
}

if comparison is not None and not comparison.get("outputs_identical_sha256", True):
    summary_payload["compatibility_issues"].append(
        "Pileup outputs differ across versions (non-identical output hash)."
    )

if {"0.2.4", "0.6.1"}.issubset(set(TEST_VERSIONS)) and "runtime_summary" in globals():
    rt = {row["version"]: row["runtime_mean_s"] for row in runtime_summary.to_dict(orient="records")}
    v024 = rt.get("0.2.4")
    v061 = rt.get("0.6.1")
    if v024 and v061:
        delta_pct = (v024 - v061) / v024 * 100.0
        if delta_pct > 0:
            summary_payload["recommendations"].append(
                f"Prefer modkit 0.6.1 for pileup runtime in this workflow (~{delta_pct:.1f}% faster on average)."
            )
        else:
            summary_payload["recommendations"].append(
                f"No runtime win from 0.6.1 observed in this benchmark ({-delta_pct:.1f}% slower)."
            )

# Flag version-specific command behavior
if "edge_case_df" in globals() and not edge_case_df.empty:
    flag_024 = edge_case_df.loc[edge_case_df["version"] == "0.2.4", "supports_force_allow_implicit"]
    flag_061 = edge_case_df.loc[edge_case_df["version"] == "0.6.1", "supports_force_allow_implicit"]
    if not flag_024.empty and not flag_061.empty and bool(flag_024.iloc[0]) and not bool(flag_061.iloc[0]):
        summary_payload["recommendations"].append(
            "Backend-aware command building is required: use --force-allow-implicit for 0.2.4 only."
        )

results_path = ROOT / "modkit_test_results.json"
with results_path.open("w") as handle:
    json.dump(summary_payload, handle, indent=2)

print("Results saved to", results_path)
print(json.dumps(summary_payload, indent=2)[:4000])
